<a href="https://colab.research.google.com/github/kirankarakalli/customer-churn-prediction-pipeline/blob/main/customer_churn_prediction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You run retention for a subscription service. Build an end-to-end ML pipeline that flags currently-active customers likely to churn in the next 30 days, so the team can target them with a retention offer. You may use any tools you like.

The data: Generate a synthetic, event-level dataset for a subscription service with at least:

an events table: user_id, timestamp, event_type (logins, usage, support tickets, payments, etc.)

a subscriptions table: user_id, signup_date, plan, monthly_price, cancel_date (nullable)

a few thousand users, a realistic churn rate (~5–15%), and activity that trails off before people cancel.

There should be no pre-made "churn" column — define and build the label yourself.

The task:

Define what "churn" means and pick a cutoff date T. Build features from data before T and the label from the window after T.

Construct the labeled training table from the raw events.

Split, train, and evaluate a model.

Report whatever metrics the retention team should actually see, and explain how you'd choose who to contact.

In [94]:
import numpy as np
import pandas as pd

In [95]:
n_users=5000

users_id=np.arange(1,n_users+1)
users_id

array([   1,    2,    3, ..., 4998, 4999, 5000])

In [96]:
signup_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2024-01-01", "2025-05-31"),
        size=n_users
    )
)

In [97]:
plans=np.random.choice(['Basic','Standard','Premium'],p=[0.3,0.5,0.2],size=n_users)
plans

array(['Premium', 'Standard', 'Standard', ..., 'Standard', 'Premium',
       'Premium'], dtype='<U8')

In [98]:
price_map={
    'Basic':10,
    'Standard':20,
    'Premium':35
}

In [99]:
subscribe_data=pd.DataFrame({
    'user_id':users_id,
    'Signup_dates':signup_dates,
    'Plans':plans,
    'monthly_price':[price_map[p] for p in plans]

})

In [100]:
subscribe_data.head()

,user_id,Signup_dates,Plans,monthly_price
0,1,2024-03-15,Premium,35
1,2,2025-02-20,Standard,20
2,3,2024-04-27,Standard,20
3,4,2024-03-25,Standard,20
4,5,2024-05-27,Standard,20


In [101]:
churn_rate=0.10

churn_users=np.random.choice(subscribe_data['user_id'],size=int(n_users*churn_rate),replace=False)

subscribe_data['cancel_date']=pd.NaT

cancel_dates=np.random.choice(pd.date_range("2025-06-01", "2025-12-31"),size=len(churn_users))

subscribe_data.loc[subscribe_data['user_id'].isin(churn_users),"cancel_date"]=cancel_dates
subscribe_data['cancel_date'].notna().mean()
subscribe_data.head()


,user_id,Signup_dates,Plans,monthly_price,cancel_date
0,1,2024-03-15,Premium,35,NaT
1,2,2025-02-20,Standard,20,NaT
2,3,2024-04-27,Standard,20,2025-08-23
3,4,2024-03-25,Standard,20,NaT
4,5,2024-05-27,Standard,20,NaT


In [102]:
# -----------------------------
# 3. Generate event-level data
# -----------------------------

events = []

for _, row in subscribe_data.iterrows():
    user_id = row["user_id"]
    signup_date = row["Signup_dates"]
    cancel_date = row["cancel_date"]

    end_date = cancel_date if pd.notna(cancel_date) else pd.Timestamp("2025-12-31")

    days_active = pd.date_range(signup_date, end_date)


    for day in days_active:
        activity_rate = 0.25

        # activity trails off before cancellation
        if pd.notna(cancel_date):
            days_before_cancel = (cancel_date - day).days

            if 0 <= days_before_cancel <= 30:
                activity_rate = 0.05
            elif 31 <= days_before_cancel <= 60:
                activity_rate = 0.12

        if np.random.rand() < activity_rate:
            events.append([user_id, day, "login"])

        if np.random.rand() < activity_rate:
            events.append([user_id, day, "usage"])

        if day.day == signup_date.day:
            events.append([user_id, day, "payment"])

        if np.random.rand() < 0.01:
            events.append([user_id, day, "support_ticket"])

events_df = pd.DataFrame(
    events,
    columns=["user_id", "timestamp", "event_type"]
)

events_df.head()

,user_id,timestamp,event_type
0,1,2024-03-15,payment
1,1,2024-03-16,usage
2,1,2024-03-17,login
3,1,2024-03-18,usage
4,1,2024-03-22,login


In [103]:
events_df.shape

(1244673, 3)

In [104]:
events_df['event_type'].value_counts()

,count
event_type,
login,573197
usage,571194
payment,77281
support_ticket,23001


In [105]:
# Define cutoff date T
T=pd.Timestamp("2025-06-01")
prediction_window=30
prediction_end=T+pd.Timedelta(days=prediction_window)

In [106]:
prediction_end

Timestamp('2025-07-01 00:00:00')

In [107]:
subscribe_data['label']=(subscribe_data['cancel_date'].between(T,prediction_end).astype(int))

In [108]:
subscribe_data['label'].value_counts()

,count
label,
0,4925
1,75


In [109]:
events_before_T=events_df[events_df['timestamp']<T].copy()

In [110]:
events_before_T.head()

,user_id,timestamp,event_type
0,1,2024-03-15,payment
1,1,2024-03-16,usage
2,1,2024-03-17,login
3,1,2024-03-18,usage
4,1,2024-03-22,login


In [111]:
feature_start_30=T-pd.Timedelta(days=30)
feature_start_90=T-pd.Timedelta(days=90)

last30=events_before_T[events_before_T['timestamp']>=feature_start_30]
last90=events_before_T[events_before_T['timestamp']>=feature_start_90]
last30.head()


,user_id,timestamp,event_type
190,1,2025-05-02,login
191,1,2025-05-05,login
192,1,2025-05-05,usage
193,1,2025-05-07,usage
194,1,2025-05-08,login


In [112]:
event_counts_30d=(last30.groupby(['user_id','event_type']).size().unstack(fill_value=0).reset_index())

In [113]:
event_counts_30d.head()

event_type,user_id,login,payment,support_ticket,usage
0,1,7,1,0,7
1,2,8,1,0,6
2,3,7,1,0,3
3,4,7,1,0,1
4,5,8,1,0,4


In [114]:
event_counts_90d=(last90.groupby(['user_id','event_type']).size().unstack(fill_value=0).reset_index())
event_counts_90d.head()

event_type,user_id,login,payment,support_ticket,usage
0,1,16,3,1,19
1,2,28,3,2,20
2,3,28,3,1,16
3,4,21,3,0,17
4,5,28,3,0,16


In [115]:
last_login=(events_before_T[events_before_T['event_type']=='login'].groupby('user_id')['timestamp'].max().reset_index())

In [116]:
last_login.head()

,user_id,timestamp
0,1,2025-05-19
1,2,2025-05-30
2,3,2025-05-28
3,4,2025-05-28
4,5,2025-05-27


In [117]:
last_login['days_since_last_login']=(T-last_login['timestamp']).dt.days

In [118]:
last_login=last_login[['user_id','days_since_last_login']]

In [119]:
last_login.head()

,user_id,days_since_last_login
0,1,13
1,2,2
2,3,4
3,4,4
4,5,5


In [120]:
subscribe_data['tenure_days']=(T-subscribe_data['Signup_dates']).dt.days

In [121]:
subscribe_data['tenure_days']=subscribe_data['tenure_days'].clip(lower=0)

In [122]:
subscribe_data.head()

,user_id,Signup_dates,Plans,monthly_price,cancel_date,label,tenure_days
0,1,2024-03-15,Premium,35,NaT,0,443
1,2,2025-02-20,Standard,20,NaT,0,101
2,3,2024-04-27,Standard,20,2025-08-23,0,400
3,4,2024-03-25,Standard,20,NaT,0,433
4,5,2024-05-27,Standard,20,NaT,0,370


In [124]:
training_df=subscribe_data[['user_id','Plans','monthly_price','tenure_days','label']].copy()

In [125]:
training_df.head()

,user_id,Plans,monthly_price,tenure_days,label
0,1,Premium,35,443,0
1,2,Standard,20,101,0
2,3,Standard,20,400,0
3,4,Standard,20,433,0
4,5,Standard,20,370,0


In [126]:
training_df=training_df.merge(event_counts_30d,on='user_id',how='left')
training_df=training_df.merge(event_counts_90d,on='user_id',how='left')
training_df=training_df.merge(last_login,on='user_id',how='left')


In [127]:
training_df.head()

,user_id,Plans,monthly_price,tenure_days,label,login_x,payment_x,support_ticket_x,usage_x,login_y,payment_y,support_ticket_y,usage_y,days_since_last_login
0,1,Premium,35,443,0,7,1,0,7,16,3,1,19,13.0
1,2,Standard,20,101,0,8,1,0,6,28,3,2,20,2.0
2,3,Standard,20,400,0,7,1,0,3,28,3,1,16,4.0
3,4,Standard,20,433,0,7,1,0,1,21,3,0,17,4.0
4,5,Standard,20,370,0,8,1,0,4,28,3,0,16,5.0


In [128]:
training_df=training_df.fillna(0)

In [129]:
training_df.head()

,user_id,Plans,monthly_price,tenure_days,label,login_x,payment_x,support_ticket_x,usage_x,login_y,payment_y,support_ticket_y,usage_y,days_since_last_login
0,1,Premium,35,443,0,7,1,0,7,16,3,1,19,13.0
1,2,Standard,20,101,0,8,1,0,6,28,3,2,20,2.0
2,3,Standard,20,400,0,7,1,0,3,28,3,1,16,4.0
3,4,Standard,20,433,0,7,1,0,1,21,3,0,17,4.0
4,5,Standard,20,370,0,8,1,0,4,28,3,0,16,5.0


In [130]:
training_df['label'].value_counts()

,count
label,
0,4925
1,75


In [131]:
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                5000 non-null   int64  
 1   Plans                  5000 non-null   object 
 2   monthly_price          5000 non-null   int64  
 3   tenure_days            5000 non-null   int64  
 4   label                  5000 non-null   int64  
 5   login_x                5000 non-null   int64  
 6   payment_x              5000 non-null   int64  
 7   support_ticket_x       5000 non-null   int64  
 8   usage_x                5000 non-null   int64  
 9   login_y                5000 non-null   int64  
 10  payment_y              5000 non-null   int64  
 11  support_ticket_y       5000 non-null   int64  
 12  usage_y                5000 non-null   int64  
 13  days_since_last_login  5000 non-null   float64
dtypes: float64(1), int64(12), object(1)
memory usage: 547.0+

In [132]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

In [144]:
X=training_df.drop(['user_id','label'],axis=1)
Y=training_df['label']
categorical_features=['Plans']
numerical_features=[col for col in X.columns if col not in categorical_features]

In [145]:
X_train,x_test,Y_train,y_test=train_test_split(X,Y,random_state=42,test_size=0.2)

In [146]:
preprocesser=ColumnTransformer(transformers=[
    ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features),
    ('num',"passthrough",numerical_features)
])

In [147]:
model=Pipeline(steps=[
    ('preprocesser',preprocesser),
    ('classifier',RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        class_weight="balanced"
    ))

])

In [148]:
print(categorical_features)

['Plans']


In [149]:
model.fit(X_train,Y_train)

Pipeline(steps=[('preprocesser',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Plans']),
                                                 ('num', 'passthrough',
                                                  ['monthly_price',
                                                   'tenure_days', 'login_x',
                                                   'payment_x',
                                                   'support_ticket_x',
                                                   'usage_x', 'login_y',
                                                   'payment_y',
                                                   'support_ticket_y',
                                                   'usage_y',
                                                   'days_since_last_login'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=8,
                                        n_estimators=200, random_state=42))])

In [151]:
y_pred = model.predict(x_test)

In [152]:
y_proba = model.predict_proba(x_test)[:, 1]

In [154]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       987
           1       0.83      0.77      0.80        13

    accuracy                           0.99      1000
   macro avg       0.92      0.88      0.90      1000
weighted avg       0.99      0.99      0.99      1000



In [155]:
print("ROC AUC:", roc_auc_score(y_test, y_proba))

ROC AUC: 0.9865170290702205


In [156]:
print("PR AUC:", average_precision_score(y_test, y_proba))

PR AUC: 0.8504488975077209


In [159]:
# -----------------------------
# 12. Decide who to contact
# -----------------------------

training_df["churn_probability"] = model.predict_proba(X)[:, 1]

top_10_percent_cutoff = training_df["churn_probability"].quantile(0.90)

retention_list = training_df[
    training_df["churn_probability"] >= top_10_percent_cutoff
].copy()

retention_list = retention_list.sort_values(
    "churn_probability",
    ascending=False
)

retention_list[
    ["user_id", "Plans", "monthly_price", "tenure_days", "churn_probability"]
].head(20)

,user_id,Plans,monthly_price,tenure_days,churn_probability
1365,1366,Standard,20,98,0.994623
4398,4399,Standard,20,204,0.993795
1715,1716,Standard,20,255,0.977676
4134,4135,Standard,20,251,0.977552
946,947,Basic,10,422,0.974889
4176,4177,Standard,20,245,0.968695
3872,3873,Standard,20,424,0.962470
723,724,Standard,20,152,0.949595
3242,3243,Premium,35,97,0.948824
851,852,Premium,35,413,0.948690
